Convert trained machine learning models into optimized and secure serialized formats (pickle/joblib/ONNX), ensuring compatibility across different platforms and environments for seamless deployment.

In [1]:
import os
import io
import pickle
import joblib
import numpy as np
import pandas as pd

In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

In [3]:
np.random.seed(42)
n_features = 8
X_raw, y_raw = make_classification(
    n_samples=1000,
    n_features=n_features,
    n_informative=5,
    n_classes=2,
    random_state=42
)

In [4]:
feature_names = [f"biomarker_{i+1}" for i in range(n_features)]
X = pd.DataFrame(X_raw, columns=feature_names)
y = y_raw

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [6]:
preprocessor = ColumnTransformer(
    transformers=[('num_scaler', StandardScaler(), feature_names)]
)

In [7]:
clinical_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

In [8]:
clinical_pipeline.fit(X_train, y_train)

c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py:136: UserWarning: Could not find the number of physical cores for the following reason:
[WinError 2] The system cannot find the file specified
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "c:\ProgramData\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 554, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "c:\ProgramData\anaconda3\Lib\subprocess.py", line 1039, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_scaler',
                                                  StandardScaler(),
                                                  ['biomarker_1', 'biomarker_2',
                                                   'biomarker_3', 'biomarker_4',
                                                   'biomarker_5', 'biomarker_6',
                                                   'biomarker_7',
                                                   'biomarker_8'])])),
                ('classifier',
                 HistGradientBoostingClassifier(random_state=42))])

In [9]:
baseline_auc = roc_auc_score(y_test, clinical_pipeline.predict_proba(X_test)[:, 1])
print(f"--- Pipeline Fit Completed ---")
print(f"Baseline Out-of-Sample ROC-AUC Score: {baseline_auc:.4f}\n")

--- Pipeline Fit Completed ---
Baseline Out-of-Sample ROC-AUC Score: 0.9767



In [10]:
joblib_filename = "clinical_diagnostic_pipeline.joblib"
joblib.dump(clinical_pipeline, joblib_filename, compress=3)

['clinical_diagnostic_pipeline.joblib']

In [11]:
reloaded_joblib_model = joblib.load(joblib_filename)
joblib_auc = roc_auc_score(y_test, reloaded_joblib_model.predict_proba(X_test)[:, 1])
print(f"[Method A: Joblib] Model successfully serialized to '{joblib_filename}'")
print(f"  • Reloaded Joblib Model ROC-AUC: {joblib_auc:.4f} (Match: {np.isclose(baseline_auc, joblib_auc)})")

[Method A: Joblib] Model successfully serialized to 'clinical_diagnostic_pipeline.joblib'
  • Reloaded Joblib Model ROC-AUC: 0.9767 (Match: True)


In [12]:
pickle_filename = "clinical_diagnostic_pipeline.pkl"
with open(pickle_filename, "wb") as f:
    pickle.dump(clinical_pipeline, f, protocol=pickle.HIGHEST_PROTOCOL)

In [13]:
with open(pickle_filename, "rb") as f:
    reloaded_pickle_model = pickle.load(f)
pickle_auc = roc_auc_score(y_test, reloaded_pickle_model.predict_proba(X_test)[:, 1])
print(f"\n[Method B: Pickle] Model successfully serialized to '{pickle_filename}'")
print(f"  • Reloaded Pickle Model ROC-AUC: {pickle_auc:.4f} (Match: {np.isclose(baseline_auc, pickle_auc)})")


[Method B: Pickle] Model successfully serialized to 'clinical_diagnostic_pipeline.pkl'
  • Reloaded Pickle Model ROC-AUC: 0.9767 (Match: True)


In [14]:
try:
    from skl2onnx import convert_sklearn
    from skl2onnx.common.data_types import FloatTensorType
    import onnxruntime as rt
    HAS_ONNX = True
except ImportError:
    HAS_ONNX = False

In [15]:
onnx_filename = "clinical_diagnostic_pipeline.onnx"

In [17]:
if HAS_ONNX:
    initial_type = [('float_input', FloatTensorType([None, len(feature_names)]))]
    
    onnx_model = convert_sklearn(clinical_pipeline, initial_types=initial_type)
    
    with open(onnx_filename, "wb") as f:
        f.write(onnx_model.SerializeToString())
        
    print(f"--- [Method C: ONNX] Model successfully exported to '{onnx_filename}' ---")
    
    sess = rt.InferenceSession(onnx_filename, providers=["CPUExecutionProvider"])
    input_name = sess.get_inputs()[0].name
    label_name = sess.get_outputs()[1].name  # Output index 1 maps to class probabilities
    
    X_test_onnx = X_test.to_numpy().astype(np.float32)
    onnx_probs_raw = sess.run([label_name], {input_name: X_test_onnx})[0]
    
    onnx_probs = np.array([p[1] for p in onnx_probs_raw])
    onnx_auc = roc_auc_score(y_test, onnx_probs)
    
    print(f"  • ONNX Runtime Model ROC-AUC : {onnx_auc:.4f}")
    print(f"  • Predictions Identical to Sklearn : {np.allclose(clinical_pipeline.predict_proba(X_test)[:, 1], onnx_probs, atol=1e-5)}")
else:
    print("--- [Method C: ONNX] Fallback Status ---")
    print("Notice: 'skl2onnx' or 'onnxruntime' is not installed in the local environment.")
    print("Native Joblib and Pickle serialization are fully operational for production deployment.")

--- [Method C: ONNX] Fallback Status ---
Notice: 'skl2onnx' or 'onnxruntime' is not installed in the local environment.
Native Joblib and Pickle serialization are fully operational for production deployment.


In [18]:
import hashlib

In [19]:
def get_artifact_metadata(filepath):
    """Computes file size in KB and cryptographic SHA-256 checksum."""
    if not os.path.exists(filepath):
        return None
    
    size_kb = os.path.getsize(filepath) / 1024.0
    
    hasher = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
            
    return {
        "File Name": filepath,
        "Size (KB)": round(size_kb, 2),
        "SHA-256 Checksum": hasher.hexdigest()
    }

In [20]:
artifacts = [joblib_filename, pickle_filename]

In [21]:
if HAS_ONNX:
    artifacts.append(onnx_filename)

df_artifacts = pd.DataFrame([get_artifact_metadata(f) for f in artifacts if os.path.exists(f)])

print("--- Production Model Artifact Integrity & Security Registry ---")
print(df_artifacts.to_string(index=False))

--- Production Model Artifact Integrity & Security Registry ---
                          File Name  Size (KB)                                                 SHA-256 Checksum
clinical_diagnostic_pipeline.joblib     148.22 fd23a981f444cde4595e4cdb4aadaa15de11020062b1c57451d1d463c7a7c077
   clinical_diagnostic_pipeline.pkl     362.31 bf55ad708e2c2503b08621cd339cfdc5a73f9d7cad5cfeef3e19b9127a7ee09e
